# Appending sharded Zarr v3 slices

This is the Zarr v3 counterpart of `zarr_v2.ipynb`. It generates 31 daily v3 inputs and creates two v3 targets: one for spatial access and one with a full-month time chunk for time-series access.

## Inner chunks and outer shards

Zarr v3 shards group several inner chunks into one storage object. The inner chunk shape controls the data read for a selection; the shard shape controls the object stored and updated. Each shard size must be a multiple of its matching chunk size.

In [1]:
import os
import shutil
from pathlib import Path

import xarray as xr
from create_inputs import make_zarr_v3_slices

from zappend.api import zappend

## Generate and collect the input slices

The generator uses exactly the same synthetic data as the v2 notebook. Only the input storage format changes: the data variables are written as Zarr v3 arrays with 450 by 450 inner chunks and 1800 by 3600 spatial shards.

In [2]:
%%time
make_zarr_v3_slices(31)

CPU times: user 4.55 s, sys: 336 ms, total: 4.89 s
Wall time: 3.14 s


In [3]:
input_dir = Path('inputs/zarr-v3')
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
slices = sorted(str(path) for path in input_dir.glob('*.zarr'))
assert len(slices) == 31
slices[:3]

['inputs/zarr-v3/slice-2024-01-01.zarr',
 'inputs/zarr-v3/slice-2024-01-02.zarr',
 'inputs/zarr-v3/slice-2024-01-03.zarr']

## Default v3 layout: favor spatial access

The first call sets `zarr_format=3` and otherwise retains the first slice's encoding, including its shard layout. It is appropriate when accessing a day or a map is more common than reading a complete point time series.

In [4]:
%%time
target = output_dir / 'v3_example_space_opt.zarr'
count = zappend(
    slices,
    target_dir=str(target),
    zarr_format=3,
    force_new=True,
    logging='INFO',
)
assert count == len(slices)

2026-11-03 09:01:34,908 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-11-03 09:01:34,909 WARNING Permanently deleting outputs/v3_example_space_opt.zarr
2026-11-03 09:01:34,920 INFO Opening slice dataset from inputs/zarr-v3/slice-2024-01-01.zarr
2026-11-03 09:01:34,934 INFO Creating target dataset outputs/v3_example_space_opt.zarr
2026-11-03 09:01:35,071 INFO Transaction completed.
2026-11-03 09:01:35,072 INFO Slice dataset inputs/zarr-v3/slice-2024-01-01.zarr closed
2026-11-03 09:01:35,083 INFO Opening slice dataset from inputs/zarr-v3/slice-2024-01-02.zarr
2026-11-03 09:01:35,096 INFO Updating target dataset outputs/v3_example_space_opt.zarr
2026-11-03 09:01:35,311 INFO Transaction completed.
2026-11-03 09:01:35,312 INFO Slice dataset inputs/zarr-v3/slice-2024-01-02.zarr closed
2026-11-03 09:01:35,324 INFO Opening slice dataset from inputs/zarr-v3/slice-2024-01-03.zarr
2026-11-03 09:01:35,334 INFO Updating target dataset outp

CPU times: user 7.02 s, sys: 804 ms, total: 7.82 s
Wall time: 5.18 s


In [5]:
%%time
ds = xr.open_zarr(target, consolidated=False)
ds

CPU times: user 17.4 ms, sys: 4.26 ms, total: 21.6 ms
Wall time: 17.2 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

## Sharded time-series layout

The second target has `(31, 90, 90)` inner chunks, so one chunk contains a full monthly point time series. The `(31, 450, 450)` shards hold 25 spatial inner chunks in each spatial direction. This reduces the number of stored objects while preserving small inner chunks for selections. For a continuing stream, choose a planned time horizon; existing Zarr chunks and shards cannot grow.

In [6]:
%%time
full_time_chunk = len(slices)
chunks = [full_time_chunk, 90, 90]
shards = [full_time_chunk, 450, 450]
target = output_dir / 'v3_example_time_opt.zarr'

count = zappend(
    slices,
    target_dir=str(target),
    zarr_format=3,
    force_new=True,
    variables={
        'temperature': {'encoding': {'chunks': chunks, 'shards': shards}},
        'quality_flag': {'encoding': {'chunks': chunks, 'shards': shards}},
        'time': {
            'encoding': {'chunks': [full_time_chunk], 'shards': [full_time_chunk]}
        },
    },
    logging='INFO',
)
assert count == full_time_chunk

2026-11-03 09:01:40,139 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-11-03 09:01:40,140 WARNING Permanently deleting outputs/v3_example_time_opt.zarr
2026-11-03 09:01:40,144 INFO Opening slice dataset from inputs/zarr-v3/slice-2024-01-01.zarr
2026-11-03 09:01:40,155 INFO Creating target dataset outputs/v3_example_time_opt.zarr
2026-11-03 09:01:41,145 INFO Transaction completed.
2026-11-03 09:01:41,146 INFO Slice dataset inputs/zarr-v3/slice-2024-01-01.zarr closed
2026-11-03 09:01:41,155 INFO Opening slice dataset from inputs/zarr-v3/slice-2024-01-02.zarr
2026-11-03 09:01:41,165 INFO Updating target dataset outputs/v3_example_time_opt.zarr
2026-11-03 09:01:42,159 INFO Transaction completed.
2026-11-03 09:01:42,160 INFO Slice dataset inputs/zarr-v3/slice-2024-01-02.zarr closed
2026-11-03 09:01:42,170 INFO Opening slice dataset from inputs/zarr-v3/slice-2024-01-03.zarr
2026-11-03 09:01:42,179 INFO Updating target dataset outputs

CPU times: user 1min 29s, sys: 6.7 s, total: 1min 36s
Wall time: 37.8 s


In [7]:
%%time
ds = xr.open_zarr(target, consolidated=False)
ds

CPU times: user 11.2 ms, sys: 1.88 ms, total: 13.1 ms
Wall time: 11.5 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

In [8]:
if os.path.exists(input_dir):
    shutil.rmtree(input_dir)